# Gradient Saliency → HDF5
Compute input-gradient saliency: how much each nucleotide position contributes to each phase prediction. Supports SmoothGrad.

In [ ]:

import sys
from pathlib import Path

# works regardless of where the notebook is opened from
PROJECT_ROOT = Path('__file__').resolve().parents[2] if '__file__' in dir() else Path('/Users/lzz/Documents/GitHub/repli-ATAC-seq/Transformer')
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import h5py
from pathlib import Path

from src.infer._utils import load_model, parse_bed, parse_region, fetch_one_hot
from src.data.data_utils import GenomeSequence


In [ ]:

# ── Parameters ────────────────────────────────────────────────────────────────
CHECKPOINT     = '/Users/lzz/Downloads/best_model.pt'
CONFIG         = str(PROJECT_ROOT / 'src/configs/transformer_wt.yaml')
FASTA          = '/Users/lzz/Desktop/repli-ATAC-seq/reference_genome/rice_all_genomes_v7.fasta'

REGION         = 'chr01:1000000-1032768'
BED            = None

SMOOTH_SAMPLES = 0
SMOOTH_NOISE   = 0.1
OUTPUT         = '/Users/lzz/Downloads/saliency.h5'


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model(CHECKPOINT, CONFIG, device)
window_size = cfg['data']['input_window_length']
genome = GenomeSequence(FASTA)
print(f'Model loaded on {device}')

if BED:
    regions = list(parse_bed(BED))
else:
    chrom, start, end = parse_region(REGION)
    regions = [(chrom, start, end, REGION, '+')]
print(f'{len(regions)} region(s)')

In [ ]:
N_PHASES = 4

def saliency_region(model, one_hot_np, device, smooth_samples, smooth_noise):
    L = one_hot_np.shape[1]
    base_t = torch.tensor(one_hot_np[None], dtype=torch.float32).to(device)

    with torch.no_grad():
        ref_pred = model(base_t)['phase_pred'].squeeze(0).cpu().numpy()

    def _compute_grad(x_t):
        grads = []
        for ph in range(N_PHASES):
            model.zero_grad()
            if x_t.grad is not None:
                x_t.grad.zero_()
            pred = model(x_t)['phase_pred']
            pred[0, ph].backward(retain_graph=(ph < N_PHASES - 1))
            grads.append(x_t.grad.detach().squeeze(0).cpu().numpy())  # [4, L]
        return np.stack(grads)  # [N_PHASES, 4, L]

    if smooth_samples > 0:
        acc = np.zeros((N_PHASES, 4, L), dtype=np.float32)
        for _ in range(smooth_samples):
            noise = torch.randn_like(base_t) * smooth_noise
            x_t = (base_t + noise).requires_grad_(True)
            acc += _compute_grad(x_t)
        grad = acc / smooth_samples
    else:
        x_t = base_t.requires_grad_(True)
        grad = _compute_grad(x_t)  # [N_PHASES, 4, L]

    grad = grad.transpose(0, 2, 1)   # [N_PHASES, L, 4]
    oh_t = one_hot_np.T[None]        # [1, L, 4]
    saliency     = grad * oh_t
    hypothetical = grad.copy()
    return ref_pred, saliency, hypothetical

In [ ]:
n, L = len(regions), window_size
Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)

with h5py.File(OUTPUT, 'w') as hf:
    hf.create_dataset('seqnames', data=np.array([r[3] for r in regions], dtype='S'))
    hf.create_dataset('chrom',    data=np.array([r[0] for r in regions], dtype='S'))
    hf.create_dataset('start',    data=np.array([r[1] for r in regions], dtype=np.int32))
    hf.create_dataset('end',      data=np.array([r[2] for r in regions], dtype=np.int32))
    hf.create_dataset('strand',   data=np.array([r[4] for r in regions], dtype='S'))
    ds_seqs = hf.create_dataset('seqs',         shape=(n, 4, L),             dtype=np.float32)
    ds_ref  = hf.create_dataset('ref_pred',     shape=(n, 4),                dtype=np.float32)
    ds_sal  = hf.create_dataset('saliency',     shape=(n, N_PHASES, L, 4),   dtype=np.float32)
    ds_hyp  = hf.create_dataset('hypothetical', shape=(n, N_PHASES, L, 4),   dtype=np.float32)

    for idx, (chrom, start, end, name, strand) in enumerate(regions):
        print(f'[{idx+1}/{n}] {name}')
        oh = fetch_one_hot(genome, chrom, start, end, window_size, strand)
        ref_pred, saliency, hypothetical = saliency_region(
            model, oh, device, SMOOTH_SAMPLES, SMOOTH_NOISE
        )
        ds_seqs[idx] = oh
        ds_ref[idx]  = ref_pred
        ds_sal[idx]  = saliency
        ds_hyp[idx]  = hypothetical

print(f'Written saliency scores to {OUTPUT}')

## Quick visualization
Sequence logo-style plot of saliency for the first region, ES phase.

In [ ]:
import matplotlib.pyplot as plt

PHASE_IDX = 0   # 0=ES, 1=MS, 2=LS, 3=G1
PHASE_NAME = ['ES', 'MS', 'LS', 'G1'][PHASE_IDX]

with h5py.File(OUTPUT, 'r') as hf:
    sal = hf['saliency'][0, PHASE_IDX]   # [L, 4]

# sum across bases to get per-position importance
importance = sal.sum(axis=1)

plt.figure(figsize=(16, 3))
plt.fill_between(range(len(importance)), importance, alpha=0.7)
plt.xlabel('Position')
plt.ylabel('Saliency (grad × input)')
plt.title(f'Gradient saliency — {PHASE_NAME} phase')
plt.tight_layout()
plt.show()